In [1]:
import torch
from transformers import BartTokenizer, BartForConditionalGeneration

2025-06-06 13:22:25.167120: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749216145.392393      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749216145.466067      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
         
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Menggunakan device: {device}\n")


# Model Path
model_paths = {
    "Baseline (BartTokenizer)": '/kaggle/input/bart-base-baseline/transformers/model-fold-1/1',
    "Indomixedtufs4": '/kaggle/input/bart-base-indomixedtufs4/transformers/model-fold-1/1',
    "Detiknews-Pemilu": '/kaggle/input/bart-base-detiknews-pemilu/transformers/model-fold-1/1',
    "Mix": '/kaggle/input/bart-base-mix/transformers/bart-base-mix/1'
}


# Fungsi meringkas artikel
def ringkas_artikel(model_path, artikel_teks, device):
    
    # Muat tokenizer
    tokenizer = BartTokenizer.from_pretrained(model_path)
    
    # Muat model dan langsung pindahkan ke device (GPU/CPU)
    model = BartForConditionalGeneration.from_pretrained(model_path).to(device)
        
    # Tokenisasi artikel dan pindahkan hasilnya (tensors) ke device
    inputs = tokenizer(artikel_teks, 
                       max_length=1024, 
                       truncation=True, 
                       return_tensors="pt").to(device)

    # Generate ringkasan
    summary_ids = model.generate(inputs["input_ids"],
                                 num_beams=4,
                                 max_length=128,
                                 min_length=12,
                                 length_penalty=1,
                                 no_repeat_ngram_size=3,
                                 early_stopping=True)

    # Decode hasil ringkasan
    ringkasan = tokenizer.decode(summary_ids[0], 
                                 skip_special_tokens=True)
    return ringkasan


Menggunakan device: cuda



In [3]:

artikel_berita = """
Komisi Pemilihan Umum Polewali Mandar mulai menggelar sosialisasi tata cara mencontreng di kalangan siswa sekolah menengah atas di Polewali Mandar , Sulawesi Barat , Jumat ( 20/2 ) . Kegiatan ini disambut antusias ratusan siswa . Namun , mereka bingung menentukan pilihan partai daftar calon legislatif yang jumlahnya mencapai 860 partai . Kegiatan sosialisasi digelar di sela-sela jam istirahat . Petugas KPU tampak kerepotan melayani pertanyaan siswa soal tata cara baru dalam pemilu . Sejumlah siswa menilai tata cara cukup rumit , terutama bagi kalangan orang lanjut usia yang buta huruf . Sosialisasi serupa juga digelar di sejumlah pusat keramaian dan tempat perbelanjaan .
"""

ringkasan_referensi = """
KPU Polewali Mandar menggelar sosialisasi tata cara Pemilu 2009 bagi kalangan siswa SMA . Sejumlah siswa menilai cara tersebut cukup rumit , terutama bagi kalangan orang lanjut usia yang buta huruf .
"""


print("--- ARTIKEL ASLI ---")
print(artikel_berita)
print("--- RINGKASAN REFERENSI ---")
print(ringkasan_referensi)

# MERINGKAS DENGAN SETIAP MODEL
for nama_model, path in model_paths.items():
    print(f"\n--- RINGKASAN DARI MODEL: {nama_model} ---")
    
    # Panggil fungsi untuk mendapatkan ringkasan dan sertakan variabel 'device'
    hasil_ringkasan = ringkas_artikel(path, artikel_berita, device)
    
    # Tampilkan hasilnya
    print(hasil_ringkasan)

--- ARTIKEL ASLI ---

Komisi Pemilihan Umum Polewali Mandar mulai menggelar sosialisasi tata cara mencontreng di kalangan siswa sekolah menengah atas di Polewali Mandar , Sulawesi Barat , Jumat ( 20/2 ) . Kegiatan ini disambut antusias ratusan siswa . Namun , mereka bingung menentukan pilihan partai daftar calon legislatif yang jumlahnya mencapai 860 partai . Kegiatan sosialisasi digelar di sela-sela jam istirahat . Petugas KPU tampak kerepotan melayani pertanyaan siswa soal tata cara baru dalam pemilu . Sejumlah siswa menilai tata cara cukup rumit , terutama bagi kalangan orang lanjut usia yang buta huruf . Sosialisasi serupa juga digelar di sejumlah pusat keramaian dan tempat perbelanjaan .

--- RINGKASAN REFERENSI ---

KPU Polewali Mandar menggelar sosialisasi tata cara Pemilu 2009 bagi kalangan siswa SMA . Sejumlah siswa menilai cara tersebut cukup rumit , terutama bagi kalangan orang lanjut usia yang buta huruf .


--- RINGKASAN DARI MODEL: Baseline (BartTokenizer) ---
Komisi Pemi